# Screener LQ45 — arus asing, outcome dan regime per emiten
## 1. Tujuan dan pertanyaan
Studi event **deskriptif eksploratoris**, bukan training/forecast. Apakah return setelah lonjakan net buy asing mengungguli IHSG pada tanggal identik? Ukur Z20 terbaru, n5/n20, mean return, hit rate, CI95 dan delta; klasifikasikan regime setiap saham.

## tl;dr
Run snapshot 24 Sep 2026: **45/45** emiten dihitung; **8** tren menguat, **33** melemah, **4** campuran. **43** emiten memiliki n20<10, sehingga interpretasi tetap provisional. Sel hasil menghitung ulang ringkasan ini saat dataset berubah. Universe adalah snapshot LQ45 saat ini; hasil bukan bukti kausalitas atau strategi tervalidasi. Tidak ada network/API call.

Roadmap: sumber → audit → EDA → Z20 dan event → alignment/cooldown → IHSG baseline → Hit Rate dan CI → regime emiten → hasil dan batas.

## 2. Imports dan konfigurasi lingkungan

In [1]:
import os, sys, json, hashlib, inspect, platform
from pathlib import Path
from collections import Counter
from statistics import mean, pstdev
import nbformat
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
SEED = 42  # rumus deterministik; tidak ada sampling acak atau tuning
pd.set_option('display.max_columns', 12)
plt.rcParams.update({'figure.figsize': (11, 5), 'font.size': 11})
print('Python', platform.python_version(), '| pandas', pd.__version__, '| seed', SEED)

Python 3.14.2 | pandas 2.2.2 | seed 42


## 3. Runtime dan guard sumber
Di Colab: mount Drive, lampirkan folder project dengan `src/` dan `data/raw/sectors/`, lalu set `IDX_EVIDENCE_LAB_ROOT`. Di Kaggle: attach salinan project sebagai dataset. Notebook tidak mengunduh data atau dependency otomatis. Paket: pandas, matplotlib, nbformat. Jangan memasukkan API key; tidak diperlukan.

In [2]:
candidates = ([Path(os.environ['IDX_EVIDENCE_LAB_ROOT'])] if os.environ.get('IDX_EVIDENCE_LAB_ROOT') else [])
candidates += [Path.cwd(), *Path.cwd().parents]
if Path('/kaggle/input').exists(): candidates += list(Path('/kaggle/input').glob('*'))
ROOT = next((p.resolve() for p in candidates if (p/'src/idx_evidence_lab').is_dir() and (p/'data/raw/sectors').is_dir()), None)
if ROOT is None: raise FileNotFoundError('Attach src/ dan data/raw/sectors/; set IDX_EVIDENCE_LAB_ROOT.')
sys.path.insert(0, str(ROOT/'src'))
from idx_evidence_lab.market_data import load_ihsg_snapshot, load_lq45_universe, load_issuer_daily, _read_snapshot
from idx_evidence_lab.market_overview_analytics import load_foreign_flow
from idx_evidence_lab.screener_analysis import analyze_bundle, analyze_issuer, wilson_interval, METHOD, VERSION
print('Runtime:', 'Colab' if 'google.colab' in sys.modules else 'Kaggle' if Path('/kaggle').exists() else 'local')
print('Root:', ROOT)

Runtime: local
Root: /Users/rasyaaudreabramantyawijaya/Downloads/Hackathon Sectors


## 4. Muat dan periksa sumber satu per satu
Loader membaca response JSON + sidecar Sectors, memeriksa hash, simbol, endpoint, konflik tanggal. Bukan mengambil API. Tabel audit tidak menyembunyikan status PARTIAL pada universe.

In [3]:
universe = load_lq45_universe(ROOT)
ihsg = load_ihsg_snapshot(ROOT)
tickers = [r['ticker'] for r in universe['symbols']]
daily = {t: load_issuer_daily(ROOT, t) for t in tickers}
foreign = load_foreign_flow(ROOT, tickers)
print('Universe:', len(tickers), universe['data_quality'], universe['issues'])
print('IHSG:', ihsg['observation_count'], ihsg['coverage_start'], ihsg['coverage_end'], ihsg['data_quality'])
print('Flow:', foreign['coverage_start'], foreign['coverage_end'], foreign['data_quality'], foreign['issues'])
for t in tickers[:3]:
    print(t, daily[t]['coverage_start'], daily[t]['coverage_end'], len(daily[t]['series']))
    display(pd.DataFrame(daily[t]['series']).head(2))
    display(pd.DataFrame(foreign['issuer_daily'][t]).head(2))

Universe: 45 PARTIAL ['Universe snapshot hash is missing or mismatched.']
IHSG: 708 2023-09-25 2026-09-24 VERIFIED
Flow: 2025-01-02 2026-09-24 VERIFIED []
AADI 2024-12-05 2026-09-24 424


,date,close,volume,open,high,low,market_cap
0,2024-12-05,6650.0,459400.0,6650,6650,6650,5.178283e+13
1,2024-12-06,7975.0,446600.0,7975,7975,7975,6.210046e+13


,date,buy,sell,net
0,2025-01-02,6.537065e+09,1.103466e+10,-4.497600e+09
1,2025-01-03,9.524080e+09,1.418408e+10,-4.659998e+09


ADMR 2023-09-25 2026-09-24 712


,date,close,volume,open,high,low,market_cap
0,2023-09-25,1435.0,34639900.0,1485,1500,1430,5.866615e+13
1,2023-09-26,1315.0,110683700.0,1440,1465,1275,5.376027e+13


,date,buy,sell,net
0,2025-01-02,4.285732e+09,5.286632e+09,-1.000901e+09
1,2025-01-03,5.067044e+09,1.486190e+09,3.580854e+09


ADRO 2023-09-25 2026-09-24 712


,date,close,volume,open,high,low,market_cap
0,2023-09-25,2950.0,32402200.0,2950,2980,2940,9.435859e+13
1,2023-09-26,2770.0,107378100.0,2960,2970,2750,8.860111e+13


,date,buy,sell,net
0,2025-01-02,3.562412e+10,1.566642e+10,1.995770e+10
1,2025-01-03,5.302627e+10,3.760343e+10,1.542284e+10


## 5. Audit schema dan kualitas
Data harga tidak diisi. Split/dividen/rights belum terverifikasi; price return bukan total return. Membership saat ini cross-check ke 45 report yang provenance-nya diverifikasi. Retain masalah metadata universe; report sekarang tidak dipakai sebagai fitur historis.

In [4]:
audit = []
for t in tickers:
    records = daily[t]['series']
    flows = foreign['issuer_daily'].get(t, [])
    report, metadata, problems = _read_snapshot(ROOT/f'data/raw/sectors/company_report/{t}/company_report.json', ROOT)
    audit.append({'ticker': t, 'daily_quality': daily[t]['data_quality'], 'prices': len(records), 'flows': len(flows),
      'missing_close': sum(r.get('close') is None for r in records),
      'duplicates': len(records)-len({r['date'] for r in records}),
      'jumps_gt40pct': sum(abs(b['close']/a['close']-1)>.4 for a,b in zip(records,records[1:])),
      'report_verified': not problems, 'current_lq45': 'LQ45' in ((report or {}).get('overview',{}).get('indices') or [])})
audit = pd.DataFrame(audit)
display(audit)
assert audit.duplicates.eq(0).all() and audit.missing_close.eq(0).all()

,ticker,daily_quality,prices,flows,missing_close,duplicates,jumps_gt40pct,report_verified,current_lq45
0,AADI,VERIFIED,424,404,0,0,0,True,True
1,ADMR,VERIFIED,712,404,0,0,0,True,True
2,ADRO,VERIFIED,712,404,0,0,0,True,True
3,AKRA,VERIFIED,712,404,0,0,0,True,True
4,AMMN,VERIFIED,712,404,0,0,0,True,True
5,AMRT,VERIFIED,712,404,0,0,0,True,True
6,ANTM,VERIFIED,712,404,0,0,0,True,True
7,ASII,VERIFIED,712,404,0,0,0,True,True
8,BBCA,VERIFIED,712,404,0,0,0,True,True
9,BBNI,VERIFIED,712,404,0,0,0,True,True


## 6. EDA
Tinjau input jumlah sesi per issuer dan net flow pada contoh alfabetis, bukan contoh yang dipilih karena hasil bagus. Lonjakan net foreign adalah observasi kanal transaksi, tidak membuktikan pemilik manfaat/niat.

In [5]:
fig, axes = plt.subplots(1, 2, figsize=(12,4))
audit[['prices','flows']].plot.hist(bins=12, alpha=.65, ax=axes[0])
axes[0].set(xlabel='Jumlah observasi / saham', ylabel='Jumlah saham', title='Cakupan 45 saham snapshot LQ45')
example = pd.DataFrame(foreign['issuer_daily'][tickers[0]])
axes[1].plot(pd.to_datetime(example.date), example.net/1e9)
axes[1].axhline(0, color='gray', lw=.8)
axes[1].set(title=f'Net asing {tickers[0]} · histori tersedia', ylabel='IDR miliar')
fig.autofmt_xdate(); plt.tight_layout(); plt.show()

<Figure size 1200x400 with 2 Axes>

## 7. Preprocessing dan feature engineering
### Context & Methods / Key assumptions
Gunakan kalender resmi IHSG. Z20 memakai **20 sesi sebelumnya**, tidak menyertakan event t. Event: net>0 dan Z>=2; entry close t+1, exit close t+1+H. Cooldown >21 sesi dipilih sebelum outcome agar tidak overlap. Semua saham memakai aturan tetap, bukan tuning. Missing flow membatalkan Z; missing harga, volume nol atau lompatan >40% membatalkan outcome. SMA memakai 200 sesi berturut-turut.

In [6]:
for name, definition in METHOD.items():
    if isinstance(definition, str): print(name, ':', definition)
# Algoritma produksi terlihat dan bisa diperiksa, bukan opaque subprocess:
print(inspect.getsource(analyze_issuer))

flow_strength : Z20 = (net asing hari t − mean net 20 sesi IHSG sebelumnya) / population std sebelumnya; bertanda, bukan skor broker.
event : Event tetap: net asing > 0 dan Z20 >= 2. Semua saham memakai aturan sama, tidak dituning pada outcome. Bukan analog dari flow terbaru yang negatif.
timing : Event diamati sesudah close t; entry close t+1; exit close t+1+H, H=5 atau 20 sesi IHSG. Availability intraday provider tidak diketahui; tidak mengklaim live execution.
overlap : Event berikutnya harus setelah exit 20D event terdahulu (jarak >21 sesi). Cooldown ditetapkan saat event, terlepas dari outcome/maturity.
sample : Sample ditampilkan n5/n20: event non-overlap dengan seluruh sesi harga tersedia, volume positif, dan benchmark tanggal identik; event belum matang tidak masuk denominator.
outcomes : 5D/20D = rata-rata simple price return saham dari entry ke exit, bruto tanpa biaya/dividen. IHSG baseline = rata-rata return indeks pada pasangan tanggal identik.
hit_rate : Hit Rate = jumlah 

## 8. Split dan normalisasi
Tidak ada supervised fit, scaler, train/test selection atau parameter dituning. Ini chronological event replay, **bukan** evaluasi model out-of-sample yang telah dipilih sebelumnya. Akhir sampel = IHSG coverage_end. Right-censored event tidak masuk denominator. Cek invariance fitur/outcome yang sudah matang saat histori dipotong.

In [7]:
cutoff = ihsg['coverage_end']
bundle = {'universe': universe, 'ihsg': ihsg, 'lq45_daily': daily, 'foreign_flow': foreign}
result = analyze_bundle(bundle, ROOT)
print('Cutoff:', cutoff, '| formula:', VERSION)
t = tickers[0]
prefix = ihsg['series'][:-60]
prefix_end = prefix[-1]['date']
short = analyze_issuer(t, [r for r in daily[t]['series'] if r['date']<=prefix_end],
 [r for r in foreign['issuer_daily'][t] if r['date']<=prefix_end], prefix)
full = next(r for r in result['rows'] if r['ticker']==t)
full_by_date = {e['event_date']: e for e in full['events']}
for e in short['events']:
    assert e['z20'] == full_by_date[e['event_date']]['z20']
    for h, outcome in e['outcomes'].items(): assert outcome == full_by_date[e['event_date']]['outcomes'][h]
print('Prefix invariance: PASS')

Cutoff: 2026-09-24 | formula: foreign-net-buy-z20-v1
Prefix invariance: PASS


## 9. Baseline
IHSG return di **entry/exit yang sama** dengan setiap event valid. Baseline Delta = return saham − return IHSG (pp), bukan alpha CAPM atau return dinormalisasi arah sinyal. Tidak menggabungkan event net sell ke studi net buy.

In [8]:
events = full['events']
e = next(e for e in events if '20' in e['outcomes'])
stock = {r['date']:r['close'] for r in daily[t]['series']}
market = {r['date']:r['price'] for r in ihsg['series']}
exit_day = e['outcomes']['20']['exit_date']; entry = e['entry_date']
stock_return = stock[exit_day]/stock[entry]-1
market_return = market[exit_day]/market[entry]-1
previous = [r['net'] for r in foreign['issuer_daily'][t] if r['date']<e['event_date']][-20:]
z_manual = (e['net']-mean(previous))/pstdev(previous)
assert abs(z_manual-e['z20'])<1e-10
assert abs(stock_return-e['outcomes']['20']['stock_return'])<1e-12
print(t, e['event_date'], 'Z=', z_manual, '| entry', entry, '| exit', exit_day)
print('Stock:', stock_return, '| IHSG:', market_return, '| delta:', stock_return-market_return)
display(pd.DataFrame([{'event_date':x['event_date'], 'entry_date':x['entry_date'], **x['outcomes'].get('20',{})} for x in events]))

AADI 2025-03-10 Z= 3.103720374270907 | entry 2025-03-11 | exit 2025-04-17
Stock: 0.041825095057034245 | IHSG: -0.01643484039505949 | delta: 0.058259935452093736


,event_date,entry_date,exit_date,stock_return,market_return,excess_return
0,2025-03-10,2025-03-11,2025-04-17,0.041825,-0.016435,0.058260
1,2025-05-08,2025-05-09,2025-06-16,0.039427,0.041680,-0.002253
2,2025-06-20,2025-06-23,2025-07-22,0.003597,0.082155,-0.078558
3,2025-08-05,2025-08-06,2025-09-04,0.021127,0.048456,-0.027329
4,2025-10-08,2025-10-09,2025-11-07,0.114379,0.017410,0.096969
5,2026-01-13,2026-01-14,2026-02-12,0.161074,-0.084940,0.246014
6,2026-02-19,2026-02-20,2026-03-27,0.150134,-0.142014,0.292148
7,2026-07-01,2026-07-02,2026-07-30,0.174051,0.076909,0.097142
8,2026-08-07,2026-08-10,2026-09-09,0.356948,0.049145,0.307803


## 10. Definisi model/statistik
Aturan deskriptif, bukan deep learning. Parameter tetap: lookback20, threshold2σ, horizon5/20, cooldown21, guard40%, SMA50/200. CI Wilson untuk **Hit Rate 20D**, bukan interval return atau distribusi skenario; interval marginal dengan asumsi independensi, tanpa koreksi multiple testing.

In [9]:
display(pd.DataFrame([{'lookback':20,'threshold_z':2,'horizons':'5,20','cooldown':21,'jump_guard':.4,'SMA':'50,200','CI':'Wilson 95%'}]))
print(inspect.getsource(wilson_interval))

,lookback,threshold_z,horizons,cooldown,jump_guard,SMA,CI
0,20,2,"5,20",21,0.4,"50,200",Wilson 95%


def wilson_interval(hits: int, count: int) -> list[float] | None:
    if not count:
        return None
    z = 1.959963984540054
    rate = hits / count
    denominator = 1 + z*z/count
    center = (rate + z*z/(2*count))/denominator
    half = z*math.sqrt(rate*(1-rate)/count + z*z/(4*count*count))/denominator
    return [max(0, center-half), min(1, center+half)]



## 11. Training
Tidak berlaku: tidak ada model terlatih, epoch, optimizer, GPU, label-based selection atau forecasting. Eksekusi berikut menghitung rumus deterministik saja.

In [10]:
result = analyze_bundle(bundle, ROOT)
table = pd.DataFrame([{k:v for k,v in r.items() if k not in ('events','regime','excluded','issues')} | {'regime':r['regime']['label']} for r in result['rows']])
print('Summary:', result['summary'])

Summary: {'issuer_count': 45, 'computed_count': 45, 'regimes': {'Tren menguat': 8, 'Tren melemah': 33, 'Tren campuran': 4}, 'small_sample_count': 43}


## 12. Validasi dan diagnostik
Rekonsiliasi denominator, hit count, benchmark delta dan batas CI untuk seluruh saham. Outcome hilang ≠ nol. Regime sekarang bukan conditioning historis pada event lama. Cek hasil lemah/negatif, bukan hanya pemenang.

In [11]:
for row in result['rows']:
    measured = [e['outcomes']['20'] for e in row['events'] if '20' in e['outcomes']]
    assert len(measured)==row['sample_20d']
    if measured:
        assert abs(row['outcome_20d']-row['baseline_20d']-row['baseline_delta_20d'])<1e-12
        assert row['hit_count']==sum(o['excess_return']>0 for o in measured)
        assert 0<=row['hit_rate_ci95'][0]<=row['hit_rate']<=row['hit_rate_ci95'][1]<=1
print('Reconciliation: PASS for',len(result['rows']),'issuers')
display(table.sort_values('baseline_delta_20d')[['ticker','sample_20d','baseline_delta_20d','hit_rate']].head(8))
display(pd.DataFrame([{'ticker':r['ticker'],**r['excluded']} for r in result['rows']]).fillna(0).head(12))

Reconciliation: PASS for 45 issuers


,ticker,sample_20d,baseline_delta_20d,hit_rate
5,AMRT,8,-0.067169,0.125000
4,AMMN,9,-0.053665,0.444444
21,GOTO,7,-0.041546,0.285714
22,HRTA,9,-0.028926,0.333333
9,BBNI,7,-0.025387,0.285714
12,BMRI,9,-0.025228,0.333333
10,BBRI,8,-0.020842,0.375000
8,BBCA,5,-0.019369,0.600000


,ticker,missing_or_constant_flow_window,overlapping_events,immature_20d,immature_5d
0,AADI,54,14,0.0,0.0
1,ADMR,54,10,0.0,0.0
2,ADRO,54,11,0.0,0.0
3,AKRA,54,12,1.0,0.0
4,AMMN,54,4,0.0,0.0
5,AMRT,54,5,1.0,1.0
6,ANTM,54,17,1.0,0.0
7,ASII,54,6,1.0,0.0
8,BBCA,54,3,1.0,0.0
9,BBNI,54,6,0.0,0.0


## 13. Results — hasil terukur, provisional
Persentase return tidak dianualisasi; delta dalam pp. n5/n20 berbeda bila outcome belum matang/tidak valid. Sampel kecil menghasilkan CI lebar. Tabel alfabetis bukan ranking rekomendasi.

In [12]:
view = table[['ticker','flow_strength_z20','sample_5d','sample_20d','outcome_5d','outcome_20d','hit_rate','baseline_delta_20d','regime']].copy()
for col in ('outcome_5d','outcome_20d','hit_rate','baseline_delta_20d'): view[col] *= 100
display(view.round(2))
print('Computed:', result['summary']['computed_count'], '| regimes:',result['summary']['regimes'], '| small samples:',result['summary']['small_sample_count'])
fig, axes = plt.subplots(1,2,figsize=(12,5))
counts=table.regime.value_counts()
axes[0].barh(counts.index, counts.values, color='#2685a8'); axes[0].set(xlabel='Jumlah emiten',title=f'Regime masing-masing saham · {cutoff}')
axes[1].hist(table.sample_20d, bins=range(0,16), color='#2685a8',edgecolor='white')
axes[1].axvline(10,color='#ad6200',linestyle='--',label='Ambang label sampel kecil')
axes[1].set(xlabel='n20 event matang non-overlap',ylabel='Jumlah saham',title='Ukuran sampel bukan jumlah sesi histori'); axes[1].legend()
plt.tight_layout(); plt.show()

/var/folders/92/6nhl2gvs7t39bfcqvytg33280000gn/T/ipykernel_26638/532640658.py:2: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  for col in ('outcome_5d','outcome_20d','hit_rate','baseline_delta_20d'): view[col] *= 100


,ticker,flow_strength_z20,sample_5d,sample_20d,outcome_5d,outcome_20d,hit_rate,baseline_delta_20d,regime
0,AADI,-0.89,9,9,1.60,11.81,66.67,11.00,Tren menguat
1,ADMR,-0.62,7,7,-1.46,7.99,57.14,8.95,Tren melemah
2,ADRO,0.05,8,8,1.05,1.12,62.50,-0.92,Tren menguat
3,AKRA,0.22,10,9,2.61,0.77,44.44,-0.66,Tren menguat
4,AMMN,-0.29,9,9,-3.91,-5.53,44.44,-5.37,Tren melemah
5,AMRT,-0.44,8,8,-0.27,-3.93,12.50,-6.72,Tren melemah
6,ANTM,-0.67,9,8,3.22,12.32,50.00,9.31,Tren melemah
7,ASII,1.05,10,9,-1.47,0.81,77.78,1.57,Tren melemah
8,BBCA,0.28,6,5,-0.37,-1.17,60.00,-1.94,Tren melemah
9,BBNI,-0.13,7,7,-0.52,0.89,28.57,-2.54,Tren melemah


Computed: 45 | regimes: {'Tren menguat': 8, 'Tren melemah': 33, 'Tren campuran': 4} | small samples: 43


<Figure size 1200x500 with 2 Axes>

In [13]:
# CI dibagi 3 panel agar 45 label tidak bertumpuk; urutan alfabetis.
fig, axes = plt.subplots(1,3,figsize=(15,7),sharex=True)
for ax, group in zip(axes, [result['rows'][i:i+15] for i in range(0,len(result['rows']),15)]):
    for i,r in enumerate(group):
        if r['hit_rate_ci95']:
            low,high=r['hit_rate_ci95']; rate=r['hit_rate']
            ax.errorbar(100*rate,i,xerr=[[100*(rate-low)],[100*(high-rate)]],fmt='o',color='#2685a8',capsize=3)
    ax.set_yticks(range(len(group)),[f"{r['ticker']} (n={r['sample_20d']})" for r in group]); ax.invert_yaxis()
    ax.axvline(50,color='gray',linestyle='--'); ax.set_xlim(0,100); ax.set_xlabel('Hit Rate ungguli IHSG 20D (%)')
    ax.grid(axis='x',alpha=.2)
fig.suptitle('CI95 Wilson per saham · event net buy Z≥2 · interval marginal, bukan prediksi')
plt.tight_layout(); plt.show()

<Figure size 1500x700 with 3 Axes>

## 14. Decision / paper replay
Tidak membuat order, sizing atau replay PnL. Return bruto event tidak sama dengan strategi setelah spread, fees, slippage, liquidity constraint. Angka bantu riset, bukan rekomendasi transaksi.

## 15. Takeaways dan batas
CI lebar menunjukkan ketidakpastian yang masih besar. Nilai negatif/Hit Rate rendah adalah hasil sah, tidak disembunyikan. Tidak ada klaim alpha tanpa adjusted prices, membership historis, biaya, dan evaluasi terpisah. Kumpulan event non-overlap mengurangi dependensi, bukan membuktikan independensi.

Backward trace: kesimpulan provisional ← tabel/CI hasil ← return/hit ratio ← kalender dan cohort non-overlap ← aturan tetap (tanpa training) ← Z20/entry delay ← audit input Sectors ← pertanyaan awal. Eksperimen selanjutnya memakai membership historis/adjusted data dan biaya; setiap pull Sectors baru memerlukan `APPROVE SECTORS API CALL` dengan scope.

In [14]:
for limitation in METHOD['limitations']: print('•', limitation)
print('Source/universe issues:', result['issues'])

• Universe LQ45 adalah membership saat ini, bukan historis; survivorship/selection bias belum dihilangkan.
• Harga mentah belum terverifikasi split/dividen/rights; return bukan total return.
• Studi event historis eksploratoris; tidak melatih model, tidak membuktikan kausalitas/alpha atau strategi siap trading. Tidak ada forecast atau order.
• Sampel <10 diberi label kecil; n=0 menghasilkan angka tidak tersedia, bukan 0%.
• CI Wilson mengasumsikan event cukup independen; cooldown mengurangi overlap, tidak menghapus serial dependence. Tidak ada multiple-testing correction.
• History flow hanya sepanjang snapshot provider yang tersedia. Data tidak real-time.
Source/universe issues: ['Universe snapshot hash is missing or mismatched.']


## 16. Reproducibility manifest
Identitas input meliputi seri IHSG, seluruh seri harga/volume dan flow, serta symbols. Fingerprint bukan pengganti audit sidecar. Notebook tidak menulis ulang dataset mentah. Hasil UI berasal dari exporter market-overview menggunakan modul yang sama.

In [15]:
print(json.dumps({'formula_version':VERSION,'as_of':cutoff,'source_fingerprint':result['source_fingerprint'],
 'module_sha256':hashlib.sha256((ROOT/'src/idx_evidence_lab/screener_analysis.py').read_bytes()).hexdigest(),
 'parameters':{'lookback':20,'z':2,'entry_delay':1,'horizons':[5,20],'cooldown':21,'jump_guard':.4},
 'seed':SEED,'python':platform.python_version(),'pandas':pd.__version__,
 'output':'docs/prototypes/market-overview-data.json → screener_analysis',
 'guards':['sources checked','no API','prefix invariance','same-date benchmark','denominator reconciliation']},indent=2))

{
  "formula_version": "foreign-net-buy-z20-v1",
  "as_of": "2026-09-24",
  "source_fingerprint": "25c3f8f654cc194518204c709220ed6586bd1571a09bfd193aab3acb18b118be",
  "module_sha256": "2f6a11e53b6df047f8772c9725225d288a88ae13b36833f56a0cd088138035a5",
  "parameters": {
    "lookback": 20,
    "z": 2,
    "entry_delay": 1,
    "horizons": [
      5,
      20
    ],
    "cooldown": 21,
    "jump_guard": 0.4
  },
  "seed": 42,
  "python": "3.14.2",
  "pandas": "2.2.2",
  "output": "docs/prototypes/market-overview-data.json \u2192 screener_analysis",
  "guards": [
    "sources checked",
    "no API",
    "prefix invariance",
    "same-date benchmark",
    "denominator reconciliation"
  ]
}
